# Generate Mistral-7B Embeddings for ArXiv Papers

This notebook runs `intfloat/e5-mistral-7b-instruct` on Google Colab's GPU.

**Steps:**
1. Upload your `arxiv_papers.db` database
2. Generate embeddings using Colab's T4/A100 GPU
3. Download the embeddings

**Requirements:** Colab with GPU runtime (T4 or A100 recommended)

In [ ]:
# Check GPU availability
!nvidia-smi

In [ ]:
# Install dependencies
!pip install -q sentence-transformers torch numpy tqdm

In [ ]:
# Upload your database file
from google.colab import files
print("Upload your arxiv_papers.db file:")
uploaded = files.upload()

In [ ]:
import sqlite3
import numpy as np
from sentence_transformers import SentenceTransformer
from tqdm import tqdm
import json
from datetime import datetime
import torch

# Configuration
MODEL_NAME = 'intfloat/e5-mistral-7b-instruct'
DB_PATH = 'arxiv_papers.db'
BATCH_SIZE = 8  # Small batch size for large model

print(f"Model: {MODEL_NAME}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")

In [ ]:
# Load papers from database
def load_papers(db_path):
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    cursor.execute("""
        SELECT id, title, abstract, primary_category
        FROM papers
        WHERE abstract IS NOT NULL AND abstract != ''
        ORDER BY id
    """)
    papers = []
    for row in cursor.fetchall():
        papers.append({
            'id': row[0],
            'title': row[1],
            'abstract': row[2],
            'primary_category': row[3]
        })
    conn.close()
    return papers

papers = load_papers(DB_PATH)
print(f"Loaded {len(papers)} papers")

In [ ]:
# Prepare texts (title + abstract)
def combine_text(title, abstract):
    if abstract:
        return f"{title}. {abstract}"
    return title

texts = [combine_text(p['title'], p['abstract']) for p in papers]
paper_ids = [p['id'] for p in papers]

print(f"Prepared {len(texts)} texts for embedding")
print(f"Sample text (truncated): {texts[0][:200]}...")

In [ ]:
# Load model
print(f"Loading {MODEL_NAME}...")
print("This may take a few minutes for the 7B model...")

model = SentenceTransformer(MODEL_NAME, trust_remote_code=True)
model = model.to('cuda' if torch.cuda.is_available() else 'cpu')

print(f"Model loaded!")
print(f"Embedding dimension: {model.get_sentence_embedding_dimension()}")

In [ ]:
# Generate embeddings
print(f"Generating embeddings for {len(texts)} texts...")
print(f"Batch size: {BATCH_SIZE}")
print(f"Estimated batches: {len(texts) // BATCH_SIZE + 1}")

start_time = datetime.now()

embeddings = model.encode(
    texts,
    batch_size=BATCH_SIZE,
    show_progress_bar=True,
    normalize_embeddings=True,
    convert_to_numpy=True
)

end_time = datetime.now()
duration = (end_time - start_time).total_seconds()

print(f"\nCompleted!")
print(f"Embeddings shape: {embeddings.shape}")
print(f"Time: {duration:.1f}s")
print(f"Rate: {len(texts)/duration:.2f} papers/s")

In [ ]:
# Save embeddings
output_name = 'intfloat-e5-mistral-7b-instruct'

# Save numpy array
np.save(f'{output_name}_embeddings.npy', embeddings)

# Save metadata
metadata = {
    'model_name': MODEL_NAME,
    'dimension': int(embeddings.shape[1]),
    'num_papers': len(papers),
    'generated_at': datetime.now().isoformat(),
    'paper_ids': paper_ids,
    'processing_time_seconds': duration,
    'papers_per_second': len(texts) / duration
}

with open(f'{output_name}_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print(f"Saved: {output_name}_embeddings.npy")
print(f"Saved: {output_name}_metadata.json")

In [ ]:
# Download the files
from google.colab import files

print("Downloading embeddings...")
files.download(f'{output_name}_embeddings.npy')
files.download(f'{output_name}_metadata.json')

print("\nDone! Move the downloaded files to:")
print("  ArXiv/data/embeddings/intfloat-e5-mistral-7b-instruct/")
print("  - Rename embeddings file to: embeddings.npy")
print("  - Rename metadata file to: metadata.json")

## After Download

Move the downloaded files to your local machine:

```bash
mkdir -p ArXiv/data/embeddings/intfloat-e5-mistral-7b-instruct/
mv intfloat-e5-mistral-7b-instruct_embeddings.npy ArXiv/data/embeddings/intfloat-e5-mistral-7b-instruct/embeddings.npy
mv intfloat-e5-mistral-7b-instruct_metadata.json ArXiv/data/embeddings/intfloat-e5-mistral-7b-instruct/metadata.json
```

Then uncomment the Mistral model in `config.py` and continue with PCA clustering.